# 04 - EDA: Shenzhen + Montgomery (Pulmonary Chest X-ray Abnormalities)

Misma estructura que los notebooks anteriores. Lo que cambia aquí: son en realidad **dos datasets clásicos distintos** empaquetados juntos (Shenzhen Hospital, China, y Montgomery County, EE. UU.), cada uno con su propia convención de nombre de archivo como etiqueta (`_0` normal / `_1` Tuberculosis) y su propio archivo de texto libre con hallazgos clínicos — de ahí se puede sacar edad/sexo por primera vez sin CSV ni DICOM, solo parseando texto.

**Por qué importa este notebook en particular**: esta es la fuente que se sospechaba desde el principio del proyecto que podía solaparse con TBX11K-simplified (ambos son datasets clásicos de TB que circulan agregados en distintos paquetes). Aquí por fin se verifica con hashes, no se sigue posponiendo.

## 0. Configuración

In [ ]:
import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "Shenzhen_Montgomery"
DATASET_DIR = Path("../Datasets/Pulmonary_chest_x_ray_adnomalities")
CHINA_DIR = DATASET_DIR / "ChinaSet_AllFiles" / "ChinaSet_AllFiles"
MONTGOMERY_DIR = DATASET_DIR / "Montgomery" / "MontgomerySet"

# El sufijo del nombre de archivo ES la etiqueta en ambos sub-datasets: _0 = normal, _1 = anormal/TB
LABEL_MAP = {"0": "Normal", "1": "Tuberculosis"}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventario básico
Se inventarían las imágenes de `CXR_png` en ambos sub-datasets. Ya se sabía por una inspección manual previa que hay un `Thumbs.db` (caché de miniaturas de Windows, no una radiografía) suelto en varias carpetas — se filtra solo, porque `eu.list_images` filtra por extensión reconocida.

In [ ]:
LABEL_RE = re.compile(r"_(\d)\.png$", re.IGNORECASE)

def inventory(cxr_dir: Path, subsource: str):
    records = []
    skipped = []
    for f in eu.list_images(cxr_dir):
        m = LABEL_RE.search(f.name)
        if not m:
            skipped.append(f.name)
            continue
        records.append({
            "subsource": subsource,
            "fname": f.name,
            "path": str(f),
            "source_label": m.group(1),
        })
    return pd.DataFrame(records), skipped

china_df, china_skipped = inventory(CHINA_DIR / "CXR_png", "Shenzhen")
monty_df, monty_skipped = inventory(MONTGOMERY_DIR / "CXR_png", "Montgomery")

print(f"Shenzhen: {len(china_df)} imágenes con etiqueta reconocida, {len(china_skipped)} archivo(s) ignorado(s): {china_skipped}")
print(f"Montgomery: {len(monty_df)} imágenes con etiqueta reconocida, {len(monty_skipped)} archivo(s) ignorado(s): {monty_skipped}")

In [ ]:
df = pd.concat([china_df, monty_df], ignore_index=True)
print(f"Total combinado: {len(df)}")
print()
print("Por sub-dataset y etiqueta de origen:")
df.groupby(["subsource", "source_label"]).size().unstack(fill_value=0)

Comparación con los números publicados de referencia (Jaeger et al. 2014, el paper original de ambos datasets): Shenzhen 662 (326 normal / 336 TB), Montgomery 138 (80 normal / 58 TB). Si coincide, confirma que el inventario por nombre de archivo es correcto.

## 2. Esquema de etiquetas y mapeo a clases objetivo

In [ ]:
df["target_class"] = df["source_label"].map(LABEL_MAP)
unmapped = df[df["target_class"].isna()]
if len(unmapped):
    print("Etiquetas sin mapear (revisar LABEL_MAP):", sorted(unmapped["source_label"].unique()))
else:
    print("Todas las imágenes quedaron mapeadas correctamente.")
df["split"] = "unassigned"  # ninguno de los dos sub-datasets trae train/val/test propio

## 3. Conteo real por clase objetivo (por sub-dataset)

In [ ]:
summary = df.groupby(["subsource", "target_class"]).size().unstack(fill_value=0)
summary["Total"] = summary.sum(axis=1)
summary.loc["Total"] = summary.sum()
summary.to_csv(OUTPUT_DIR / "conteo_por_clase.csv")
summary

In [ ]:
totales = df["target_class"].value_counts().sort_index()
ax = totales.plot(kind="bar", figsize=(6, 4), color="#937860")
ax.set_title(f"{DATASET_NAME}: total de imágenes por clase")
ax.set_ylabel("Número de imágenes")
for i, v in enumerate(totales):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase.png", dpi=120)
plt.show()

## 4. Calidad e integridad

In [ ]:
probe_results = [eu.probe_image(p) for p in tqdm(df["path"], desc="Verificando imágenes")]
df["width"], df["height"], df["mode"], df["error"] = zip(*probe_results)
corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no abribles: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Modo de color:")
display(df["mode"].value_counts())
print()
print("Dimensiones por sub-dataset (Shenzhen y Montgomery usan equipos distintos, se espera que difieran):")
df.groupby("subsource")[["width", "height"]].describe()

### 4.1 Candidatas a excluir por calidad de imagen (mismo criterio que las fuentes anteriores)

In [ ]:
calidad = eu.flag_quality_issues(df[df["error"].isna()], desc=DATASET_NAME)
sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
sospechosas.groupby(["subsource", "target_class"]).size()

In [ ]:
sospechosas[["path", "subsource", "target_class", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)
eu.show_flagged(sospechosas, label_cols=("subsource", "target_class", "motivo"), save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png")

## 5. Muestra visual por clase

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png")

### 5.1 Bonus: máscara pulmonar de Montgomery (izquierda + derecha combinadas)
A diferencia de COVID-QU-Ex (una sola máscara pulmonar), Montgomery trae el pulmón izquierdo y derecho como máscaras **separadas** — se combinan aquí solo para visualizar.

In [ ]:
import numpy as np

ejemplo_monty = monty_df.sample(1, random_state=RANDOM_SEED).iloc[0]
left_mask_path = MONTGOMERY_DIR / "ManualMask" / "leftMask" / ejemplo_monty["fname"]
right_mask_path = MONTGOMERY_DIR / "ManualMask" / "rightMask" / ejemplo_monty["fname"]

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
with Image.open(ejemplo_monty["path"]) as img:
    axes[0].imshow(img, cmap="gray")
axes[0].set_title("Imagen"); axes[0].axis("off")

if left_mask_path.exists() and right_mask_path.exists():
    left = np.array(Image.open(left_mask_path).convert("L"))
    right = np.array(Image.open(right_mask_path).convert("L"))
    combined = np.maximum(left, right)
    axes[1].imshow(combined, cmap="gray")
    axes[1].set_title("Máscara pulmonar (izq + der)")
else:
    axes[1].set_title("Máscara no encontrada")
axes[1].axis("off")
plt.suptitle(f"Montgomery - {ejemplo_monty['fname']}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ejemplo_imagen_y_mascara.png", dpi=120)
plt.show()

## 6. Riesgo de solapamiento / duplicados
### 6.1 Duplicados dentro de este mismo dataset

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
df["phash"] = [eu.phash_file(p) for p in tqdm(df["path"], desc="Hash perceptual")]
print(f"Duplicados exactos (md5) dentro de este dataset: {df['md5'].duplicated().sum()}")
print(f"Duplicados perceptuales (phash) dentro de este dataset: {df['phash'].duplicated().sum()}")

### 6.2 La pregunta pendiente desde el inicio del proyecto: ¿se solapa con TBX11K-simplified?

Se cruza el `phash` de este dataset contra el `phash` ya guardado de TBX11K-simplified (`01_eda_tbx11k.ipynb`, Sección 6). El phash detecta la misma imagen aunque haya sido reescalada, recomprimida o convertida a otro formato entre un paquete de Kaggle y otro — un MD5 exacto no serviría para esto.

In [ ]:
tbx11k_hashes_path = HASH_DIR / "TBX11K-simplified_md5.csv"
if tbx11k_hashes_path.exists():
    tbx11k_hashes = pd.read_csv(tbx11k_hashes_path)
    overlap_phash = set(df["phash"]) & set(tbx11k_hashes["phash"])
    overlap_rows = df[df["phash"].isin(overlap_phash)]
    print(f"Imágenes de Shenzhen/Montgomery cuyo phash YA aparece en TBX11K-simplified: "
          f"{len(overlap_rows)} de {len(df)} ({100 * len(overlap_rows) / len(df):.1f}%)")
    print()
    print("Por sub-dataset y clase:")
    display(overlap_rows.groupby(["subsource", "target_class"]).size())
else:
    print(f"No se encontró {tbx11k_hashes_path} — correr primero 01_eda_tbx11k.ipynb.")

In [ ]:
df[["path", "md5", "phash", "subsource", "target_class", "split"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_md5.csv", index=False
)

## 7. Metadata demográfica / poblacional

Sin CSV ni DICOM, pero cada imagen trae un `.txt` de hallazgos clínicos en texto libre — se parsea con expresiones regulares (formato distinto en cada sub-dataset: Shenzhen es texto libre tipo "male 45yrs", Montgomery es más estructurado tipo "Patient's Sex: F / Patient's Age: 027Y").

In [ ]:
SEX_RE = re.compile(r"(male|female|[MF])\b", re.IGNORECASE)
AGE_RE = re.compile(r"(\d{1,3})\s*(?:yrs?|Y)\b", re.IGNORECASE)

def clinical_reading_dir(subsource):
    return (CHINA_DIR if subsource == "Shenzhen" else MONTGOMERY_DIR) / "ClinicalReadings"

def parse_clinical_reading(row):
    txt_path = clinical_reading_dir(row["subsource"]) / row["fname"].replace(".png", ".txt")
    if not txt_path.exists():
        return pd.Series({"sex": None, "age": None, "findings_text": None})
    text = txt_path.read_text(errors="ignore")
    sex_match = SEX_RE.search(text)
    age_match = AGE_RE.search(text)
    sex = sex_match.group(1)[0].upper() if sex_match else None
    age = int(age_match.group(1)) if age_match else None
    return pd.Series({"sex": sex, "age": age, "findings_text": text.strip().replace("\n", " | ")})

meta = df.progress_apply(parse_clinical_reading, axis=1)
df = pd.concat([df, meta], axis=1)

print(f"Sexo no reconocido: {df['sex'].isna().sum()} de {len(df)}")
print(f"Edad no reconocida: {df['age'].isna().sum()} de {len(df)}")

In [ ]:
print("Edad por sub-dataset:")
display(df.groupby("subsource")["age"].describe())
print()
print("Sexo por sub-dataset:")
pd.crosstab(df["subsource"], df["sex"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.boxplot(column="age", by="subsource", ax=axes[0]); axes[0].set_title("Edad por sub-dataset")
pd.crosstab(df["subsource"], df["sex"]).plot(kind="bar", ax=axes[1]); axes[1].set_title("Sexo por sub-dataset")
plt.suptitle("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "demografia_por_clase.png", dpi=120)
plt.show()

El texto libre de los casos de Tuberculosis usa abreviaturas radiológicas variadas (`TB`, `STB` -secondary TB-, `PTB` -primary TB-, `ATB` -active TB-, `NATB`, etc.) — no es una taxonomía consistente como el `tb_type` (active/latent) de TBX11K, así que aquí se trata solo como texto libre de referencia, no como una subcategoría clínica más para el proyecto.

## 8. Veredicto

- Inventario confirmado contra los números publicados de referencia: Shenzhen 662 (326/336), Montgomery 138 (80/58) — 800 imágenes en total, Normal 406 / Tuberculosis 394.
- **Resultado del cruce con TBX11K-simplified** (Sección 6.2, completar con el número real de la ejecución): esto resuelve la duda que se venía posponiendo desde que se detectó como riesgo al incorporar esta fuente al plan. Si el solapamiento es alto, esta fuente aporta poco Tuberculosis "nuevo" más allá de lo que ya trae TBX11K; si es bajo, sí suma un techo adicional real.
- Primera fuente sin CSV ni DICOM que aun así permite extraer edad/sexo, parseando texto libre — con calidad de parseo variable entre Shenzhen (texto libre) y Montgomery (más estructurado).
- Montgomery aporta máscaras de pulmón izquierdo/derecho por separado (distinto de la máscara única de COVID-QU-Ex) — dato adicional para preprocesamiento/explicabilidad si se decide usarlas.
- Se encontró y excluyó automáticamente un archivo `Thumbs.db` (caché de Windows) en varias carpetas — no es una radiografía, ya filtrado por extensión.